<a href="https://colab.research.google.com/github/Jiaah116/DNSC6330-10-Individual-Week-1-Compas-analysis-python/blob/main/DNSC6330_10%E3%80%80Compas_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**DNSC 6330-10 Weekly individual Assignment 1**

*   GWU ID: 29919940
*   Name: Yukari "Jiah" Teranishi
*   Deadline:March 30th 2026 12:00PM
*   Notes: Due to parents emergency, olease allow me submit this late. I am so sorry.


# Compas Analysis

What follows are the calculations performed for ProPublica's analaysis of the COMPAS Recidivism Risk Scores. It might be helpful to open [the methodology](https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm/) in another tab to understand the following.

## Loading the Data

We select fields for severity of charge, number of priors, demographics, age, sex, compas scores, and whether each person was accused of a crime within two years.

In [ ]:
import pandas as pd

In [ ]:
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw_data = pd.read_csv(url)
# Load the COMPAS dataset used in the original R script.
# ensuring consistency with the lecture workflow

In [ ]:
print(raw_data.shape)
raw_data.head() #Data Ceaning and prepartion

In [ ]:
import pandas as pd

# Load the dataset from the given URL
raw_data = pd.read_csv(url)

# Filter the dataset to match the original analysis conditions
# - Keep rows where screening date is within 30 days of arrest
# - Remove rows where recidivism info is missing (is_recid = -1)
# - Remove minor traffic offenses (c_charge_degree = "O")
# - Remove rows with missing score_text
df = raw_data[
    (raw_data["days_b_screening_arrest"] >= -30) &
    (raw_data["days_b_screening_arrest"] <= 30) &
    (raw_data["is_recid"] != -1) &
    (raw_data["c_charge_degree"] != "O") &
    (raw_data["score_text"] != "N/A")
].copy()

# Create a copy of the cleaned dataset
df_main = df.copy()

# Check the column names to understand what variables are available
print(df_main.columns.tolist())

In [ ]:
# Convert jail dates to datetime format
df["c_jail_in"] = pd.to_datetime(df["c_jail_in"])
df["c_jail_out"] = pd.to_datetime(df["c_jail_out"])

# Compute length of stay (in days)
df["length_of_stay"] = (df["c_jail_out"] - df["c_jail_in"]).dt.days

# Calculate correlation with COMPAS decile score
correlation = df["length_of_stay"].corr(df["decile_score"])

print("Correlation:", correlation)

In [ ]:
# Select only relevant columns (optional but cleaner)
df_model = df[[
    "age", "c_charge_degree", "race", "age_cat",
    "score_text", "sex", "priors_count",
    "days_b_screening_arrest", "decile_score",
    "is_recid", "two_year_recid"
]].copy()

In [ ]:
# Create binary target (High vs Low/Medium)
df_model["score_factor"] = df_model["score_text"].apply(
    lambda x: "HighScore" if x != "Low" else "LowScore"
)

In [ ]:
# Convert categorical variables
df_model["crime_factor"] = df_model["c_charge_degree"].astype("category")
df_model["age_factor"] = df_model["age_cat"].astype("category")
df_model["race_factor"] = df_model["race"].astype("category")
df_model["gender_factor"] = df_model["sex"].astype("category")


In [ ]:
# Check result
df_model.head()

## Exploratory Data Analysis (EDA)

In this section, I take a closer look at the dataset to get a general sense of how the data is distributed. I focus on variables like risk scores, gender, and race to understand the overall structure before moving on to modeling. This step is helpful for spotting any imbalances or patterns that might affect later results.

In [ ]:
print(df["score_text"].value_counts())

In [ ]:
print(pd.crosstab(df["sex"], df["race"]))

In [ ]:
# Check gender distribution
print(df["sex"].value_counts())
print(df["sex"].value_counts(normalize=True) * 100)

In [ ]:
race_counts = df["race"].value_counts()
race_percent = race_counts / len(df) * 100

# Print percentages for each race group
print("Black defendants: %.2f%%" % race_percent["African-American"])
print("White defendants: %.2f%%" % race_percent["Caucasian"])
print("Hispanic defendants: %.2f%%" % race_percent["Hispanic"])
print("Asian defendants: %.2f%%" % race_percent["Asian"])
print("Native American defendants: %.2f%%" % race_percent["Native American"])

In [ ]:
# how many
recid_count = (df["two_year_recid"] == 1).sum()
print("Recidivism count:", recid_count)

# ratio
recid_rate = recid_count / len(df) * 100
print("Recidivism rate:", recid_rate)

### Key Observations

- Most defendants are classified as "Low" risk, followed by "Medium" and "High".
- The dataset is heavily skewed toward male defendants.
- African-American defendants make up the largest proportion of the dataset (~51%).
- These distributions are important when interpreting fairness and bias in the model.

Judges are often presented with two sets of scores from the Compas system -- one that classifies people into High, Medium and Low risk, and a corresponding decile score. There is a clear downward trend in the decile scores as those scores increase for white defendants.

In [ ]:
import statsmodels.formula.api as smf

# Make a fresh copy from the filtered dataframe
df_glm = df.copy()

# Create categorical variables with the same reference groups as the R code
df_glm["crime_factor"] = pd.Categorical(df_glm["c_charge_degree"])
df_glm["age_factor"] = pd.Categorical(
    df_glm["age_cat"],
    categories=["25 - 45", "Greater than 45", "Less than 25"]
)
df_glm["race_factor"] = pd.Categorical(
    df_glm["race"],
    categories=["Caucasian", "African-American", "Asian", "Hispanic", "Native American", "Other"]
)
df_glm["gender_factor"] = pd.Categorical(
    df_glm["sex"],
    categories=["Male", "Female"]
)

# Binary target: 1 if Medium/High, 0 if Low
df_glm["score_factor"] = (df_glm["score_text"] != "Low").astype(int)

# Logistic regression equivalent to the R glm
model_glm = smf.logit(
    "score_factor ~ C(gender_factor) + C(age_factor) + C(race_factor) + priors_count + C(crime_factor) + two_year_recid",
    data=df_glm
).fit()

print(model_glm.summary())

In [ ]:
import numpy as np

odds_ratios = pd.DataFrame({
    "Coefficient": model_glm.params,
    "Odds Ratio": np.exp(model_glm.params)
})
print(odds_ratios)

## Racial Bias in Compas

After filtering out bad rows, our first question is whether there is a significant difference in Compas scores between races. To do so we need to change some variables into factors, and run a logistic regression, comparing low scores to high scores.

In [ ]:
# Convert selected coefficients into more interpretable ratios
control = np.exp(model_glm.params["Intercept"]) / (1 + np.exp(model_glm.params["Intercept"]))

black_effect = np.exp(model_glm.params["C(race_factor)[T.African-American]"]) / (
    1 - control + control * np.exp(model_glm.params["C(race_factor)[T.African-American]"])
)

female_effect = np.exp(model_glm.params["C(gender_factor)[T.Female]"]) / (
    1 - control + control * np.exp(model_glm.params["C(gender_factor)[T.Female]"])
)

under25_effect = np.exp(model_glm.params["C(age_factor)[T.Less than 25]"]) / (
    1 - control + control * np.exp(model_glm.params["C(age_factor)[T.Less than 25]"])
)

print("Black defendants relative effect:", black_effect)
print("Female defendants relative effect:", female_effect)
print("Under-25 defendants relative effect:", under25_effect)

Black defendants are 45% more likely than white defendants to receive a higher score correcting for the seriousness of their crime, previous arrests, and future criminal behavior.

In [ ]:
print("Black defendants relative effect:", black_effect)

Women are 19.4% more likely than men to get a higher score.

In [ ]:
print("Female defendants relative effect:", female_effect)

Most surprisingly, people under 25 are 2.5 times as likely to get a higher score as middle aged defendants.

In [ ]:
print("Under-25 defendants relative effect:", under25_effect)

### Risk of Violent Recidivism

Compas also offers a score that aims to measure a persons risk of violent recidivism, which has a similar overall accuracy to the Recidivism score. As before, we can use a logistic regression to test for racial bias.

In [ ]:
raw_data = pd.read_csv(
    "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years-violent.csv"
)
print(len(raw_data))

In [ ]:

df = raw_data[[
    "age", "c_charge_degree", "race", "age_cat", "v_score_text", "sex",
    "priors_count", "days_b_screening_arrest", "v_decile_score",
    "is_recid", "two_year_recid"
]].copy()

df = df[
    (df["days_b_screening_arrest"] <= 30) &
    (df["days_b_screening_arrest"] >= -30) &
    (df["is_recid"] != -1) &
    (df["c_charge_degree"] != "O") &
    (df["v_score_text"] != "N/A")
].copy()

print(len(df))

In [ ]:
print(df["age_cat"].value_counts())

In [ ]:
print(df["race"].value_counts())

In [ ]:
print(df["v_score_text"].value_counts())

In [ ]:
print((df["two_year_recid"] == 1).mean() * 100)

In [ ]:
print((df["two_year_recid"] == 1).sum())

In [ ]:
import matplotlib.pyplot as plt

black_counts = (
    df[df["race"] == "African-American"]["v_decile_score"]
    .value_counts()
    .sort_index()
)

white_counts = (
    df[df["race"] == "Caucasian"]["v_decile_score"]
    .value_counts()
    .sort_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(black_counts.index.astype(str), black_counts.values)
axes[0].set_title("Black Defendant's Violent Decile Scores")
axes[0].set_xlabel("Violent Decile Score")
axes[0].set_ylabel("count")
axes[0].set_ylim(0, 700)

axes[1].bar(white_counts.index.astype(str), white_counts.values)
axes[1].set_title("White Defendant's Violent Decile Scores")
axes[1].set_xlabel("Violent Decile Score")
axes[1].set_ylabel("count")
axes[1].set_ylim(0, 700)

plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

violent_glm = df.copy()

violent_glm["crime_factor"] = pd.Categorical(violent_glm["c_charge_degree"])
violent_glm["age_factor"] = pd.Categorical(
    violent_glm["age_cat"],
    categories=["25 - 45", "Greater than 45", "Less than 25"]
)
violent_glm["race_factor"] = pd.Categorical(
    violent_glm["race"],
    categories=["Caucasian", "African-American", "Asian", "Hispanic", "Native American", "Other"]
)
violent_glm["gender_factor"] = pd.Categorical(
    violent_glm["sex"],
    categories=["Male", "Female"]
)

violent_glm["score_factor"] = (violent_glm["v_score_text"] != "Low").astype(int)

violent_model = smf.logit(
    "score_factor ~ C(gender_factor) + C(age_factor) + C(race_factor) + priors_count + C(crime_factor) + two_year_recid",
    data=violent_glm
).fit()

print(violent_model.summary())

violent_odds = pd.DataFrame({
    "Coefficient": violent_model.params,
    "Odds Ratio": np.exp(violent_model.params)
})
print(violent_odds)

The violent score overpredicts recidivism for black defendants by 77.3% compared to white defendants.

Defendands under 25 are 7.4 times as likely to get a higher score as middle aged defendants.

In [ ]:
violent_control = np.exp(violent_model.params["Intercept"]) / (
    1 + np.exp(violent_model.params["Intercept"])
)

violent_black_effect = np.exp(
    violent_model.params["C(race_factor)[T.African-American]"]
) / (
    1 - violent_control +
    violent_control * np.exp(violent_model.params["C(race_factor)[T.African-American]"])
)

violent_under25_effect = np.exp(
    violent_model.params["C(age_factor)[T.Less than 25]"]
) / (
    1 - violent_control +
    violent_control * np.exp(violent_model.params["C(age_factor)[T.Less than 25]"])
)

print(f"Black defendants relative effect: {violent_black_effect:.6f}")
print(f"Under-25 defendants relative effect: {violent_under25_effect:.6f}")

## Predictive Accuracy of COMPAS

In order to test whether Compas scores do an accurate job of deciding whether an offender is Low, Medium or High risk,  we ran a Cox Proportional Hazards model. Northpointe, the company that created COMPAS and markets it to Law Enforcement, also ran a Cox model in their [validation study](http://cjb.sagepub.com/content/36/1/21.abstract).

We used the counting model and removed people when they were incarcerated. Due to errors in the underlying jail data, we need to filter out 32 rows that have an end date more than the start date. Considering that there are 13,334 total rows in the data, such a small amount of errors will not affect the results.

In [ ]:
!pip install lifelines

In [ ]:
grp = df_main.drop_duplicates(subset="id")
print(len(grp))

The number of unique individuals differs from the original R analysis because a different dataset is used.
After filtering, the dataset contains 6172 unique individuals with no duplicates.


In [ ]:
from lifelines import CoxPHFitter

cox_data = df_main.copy()
cox_data["duration"] = abs(cox_data["days_b_screening_arrest"]) + 1
cox_data["event"] = cox_data["two_year_recid"]
cox_data["score_factor"] = (cox_data["score_text"] != "Low").astype(int)

cox_data = cox_data[["duration", "event", "score_factor"]]

cph = CoxPHFitter()
cph.fit(cox_data, duration_col="duration", event_col="event")
cph.print_summary()

In [ ]:
import pandas as pd

cox_data = df_main.copy()
cox_data["duration"] = abs(cox_data["days_b_screening_arrest"]) + 1
cox_data["event"] = cox_data["two_year_recid"]

# 3カテゴリにする
cox_data["score_factor"] = pd.Categorical(
    cox_data["score_text"],
    categories=["Low", "Medium", "High"]
)

# ダミー変数化（Lowを基準）
cox_data = pd.get_dummies(cox_data, columns=["score_factor"], drop_first=True)

cox_data = cox_data[["duration", "event", "score_factor_Medium", "score_factor_High"]]

from lifelines import CoxPHFitter
cph = CoxPHFitter()
cph.fit(cox_data, duration_col="duration", event_col="event")
cph.print_summary()

In [ ]:
grp = df_main.drop_duplicates(subset="id")
print(grp["race"].value_counts())

In [ ]:
import pandas as pd

cox_url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/cox-parsed.csv"
data = pd.read_csv(cox_url)

# Remove missing score_text rows and keep only valid counting-process rows
data = data[data["score_text"].notna() & (data["end"] > data["start"])].copy()

data["race_factor"] = pd.Categorical(
    data["race"],
    categories=["Caucasian", "African-American", "Asian", "Hispanic", "Native American", "Other"]
)

data["score_factor"] = pd.Categorical(
    data["score_text"],
    categories=["Low", "High", "Medium"]
)

grp = data.drop_duplicates(subset="id").copy()

print(len(grp))
print(grp["score_factor"].value_counts(dropna=False))
print(grp["race_factor"].value_counts(dropna=False))

People placed in the High category are 3.5 times as likely to recidivate, and the COMPAS system's concordance 63.6%. This is lower than the accuracy quoted in the Northpoint study of 68%.

In [ ]:
from lifelines import CoxTimeVaryingFitter

cox_data = data[["id", "start", "end", "event", "score_factor"]].copy()
cox_data = pd.get_dummies(cox_data, columns=["score_factor"], drop_first=True)

ctv = CoxTimeVaryingFitter()
ctv.fit(
    cox_data,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

ctv.print_summary()

In [ ]:
from lifelines import CoxTimeVaryingFitter

# Cox model using decile score
decile_data = data[["id", "start", "end", "event", "decile_score"]].copy()

ctv_decile = CoxTimeVaryingFitter()
ctv_decile.fit(
    decile_data,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

ctv_decile.print_summary()

The Cox model using the COMPAS decile score shows that each one-point increase in decile score is associated with an increase in recidivism risk. The hazard ratio is approximately 1.22, meaning that for every additional point in the decile score, the risk of recidivism increases by about 22%.

This result is consistent with the original R analysis and suggests that the decile score is a meaningful predictor of recidivism.

COMPAS's decile scores are a bit more accurate at 66%.

We can test if the algorithm is behaving differently across races by including a race interaction term in the cox model.

In [ ]:
from lifelines import CoxTimeVaryingFitter
import pandas as pd

# Keep only the columns needed for the interaction model
interaction_data = data[["id", "start", "end", "event", "race_factor", "score_factor"]].copy()

# Create dummy variables
interaction_data = pd.get_dummies(
    interaction_data,
    columns=["race_factor", "score_factor"],
    drop_first=True
)

# Create interaction terms
interaction_data["AA_High"] = interaction_data["race_factor_African-American"] * interaction_data["score_factor_High"]
interaction_data["AA_Medium"] = interaction_data["race_factor_African-American"] * interaction_data["score_factor_Medium"]

interaction_data["Asian_High"] = interaction_data["race_factor_Asian"] * interaction_data["score_factor_High"]
interaction_data["Asian_Medium"] = interaction_data["race_factor_Asian"] * interaction_data["score_factor_Medium"]

interaction_data["Hispanic_High"] = interaction_data["race_factor_Hispanic"] * interaction_data["score_factor_High"]
interaction_data["Hispanic_Medium"] = interaction_data["race_factor_Hispanic"] * interaction_data["score_factor_Medium"]

interaction_data["NativeAmerican_High"] = interaction_data["race_factor_Native American"] * interaction_data["score_factor_High"]
interaction_data["NativeAmerican_Medium"] = interaction_data["race_factor_Native American"] * interaction_data["score_factor_Medium"]

interaction_data["Other_High"] = interaction_data["race_factor_Other"] * interaction_data["score_factor_High"]
interaction_data["Other_Medium"] = interaction_data["race_factor_Other"] * interaction_data["score_factor_Medium"]

# Fit counting-process Cox model
ctv_interaction = CoxTimeVaryingFitter()
ctv_interaction.fit(
    interaction_data,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

ctv_interaction.print_summary()

The interaction term shows a similar disparity as the logistic regression above.

High risk white defendants are 3.61 more likely than low risk white defendants, while High risk black defendants are 2.99 more likely than low.

In [ ]:
import math
print("Black High Hazard: %.2f" % (math.exp(-0.18976 + 1.28350)))
print("White High Hazard: %.2f" % (math.exp(1.28350)))
print("Black Medium Hazard: %.2f" % (math.exp(0.84286 - 0.17261)))
print("White Medium Hazard: %.2f" % (math.exp(0.84286)))

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

plt.figure(figsize=(8,6))

for label in data["score_factor"].unique():
    subset = data[data["score_factor"] == label]
    kmf.fit(
        durations=subset["end"] - subset["start"],
        event_observed=subset["event"],
        label=f"score_factor={label}"
    )
    kmf.plot_survival_function()

plt.title("Overall")
plt.ylim(0,1)
plt.xlabel("Time")
plt.ylabel("Survival Probability")
plt.show()

Black defendants do recidivate at higher rates according to race specific Kaplan Meier plots.

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

# White
white = data[data["race"] == "Caucasian"]

plt.figure()

for group in ["Low", "Medium", "High"]:
    subset = white[white["score_text"] == group]
    kmf.fit(subset["end"], event_observed=subset["event"], label=group)
    kmf.plot()

plt.title("White defendants")
plt.xlabel("time")
plt.ylabel("survival")
plt.show()


# Black
black = data[data["race"] == "African-American"]

plt.figure()

for group in ["Low", "Medium", "High"]:
    subset = black[black["score_text"] == group]
    kmf.fit(subset["end"], event_observed=subset["event"], label=group)
    kmf.plot()

plt.title("Black defendants")
plt.xlabel("time")
plt.ylabel("survival")
plt.show()

In [ ]:
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()

# Overall
print("=== Overall ===")
for group in ["Low", "Medium", "High"]:
    subset = data[data["score_text"] == group]
    kmf.fit(subset["end"], event_observed=subset["event"])
    surv = kmf.survival_function_at_times(730)
    print(f"{group}: {float(surv.iloc[0]):.3f}")

# White
print("\n=== White defendants ===")
white = data[data["race"] == "Caucasian"]

for group in ["Low", "Medium", "High"]:
    subset = white[white["score_text"] == group]
    kmf.fit(subset["end"], event_observed=subset["event"])
    surv = kmf.survival_function_at_times(730)
    print(f"{group}: {float(surv.iloc[0]):.3f}")

# Black
print("\n=== Black defendants ===")
black = data[data["race"] == "African-American"]

for group in ["Low", "Medium", "High"]:
    subset = black[black["score_text"] == group]
    kmf.fit(subset["end"], event_observed=subset["event"])
    surv = kmf.survival_function_at_times(730)
    print(f"{group}: {float(surv.iloc[0]):.3f}")

Race specific models have similar concordance values.

In [ ]:
from lifelines import CoxTimeVaryingFitter
import pandas as pd

# White-only Cox model
white_data = data[data["race"] == "Caucasian"].copy()
white_data["score_factor"] = pd.Categorical(
    white_data["score_text"],
    categories=["Low", "High", "Medium"]
)

white_cox = white_data[["id", "start", "end", "event", "score_factor"]].copy()
white_cox = pd.get_dummies(white_cox, columns=["score_factor"], drop_first=True)

ctv_white = CoxTimeVaryingFitter()
ctv_white.fit(
    white_cox,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

print("=== White defendants ===")
ctv_white.print_summary()


# Black-only Cox model
black_data = data[data["race"] == "African-American"].copy()
black_data["score_factor"] = pd.Categorical(
    black_data["score_text"],
    categories=["Low", "High", "Medium"]
)

black_cox = black_data[["id", "start", "end", "event", "score_factor"]].copy()
black_cox = pd.get_dummies(black_cox, columns=["score_factor"], drop_first=True)

ctv_black = CoxTimeVaryingFitter()
ctv_black.fit(
    black_cox,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

print("\n=== Black defendants ===")
ctv_black.print_summary()

Compas's violent recidivism score has a slightly higher overall concordance score of 65.1%.

In [ ]:
# Load violent dataset
violent_url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/cox-violent-parsed.csv"
violent_data = pd.read_csv(violent_url)

# Filter (same logic as before)
violent_data = violent_data[
    (violent_data["score_text"].notna()) &
    (violent_data["end"] > violent_data["start"])
].copy()

# Create categorical variables
violent_data["score_factor"] = pd.Categorical(
    violent_data["score_text"],
    categories=["Low", "High", "Medium"]
)

# Drop duplicate ids (same as grp in R)
violent_grp = violent_data.drop_duplicates(subset="id").copy()
print(len(violent_grp))

# Prepare Cox data
violent_cox = violent_data[["id", "start", "end", "event", "score_factor"]].copy()
violent_cox = pd.get_dummies(violent_cox, columns=["score_factor"], drop_first=True)

# Fit model
ctv_violent = CoxTimeVaryingFitter()
ctv_violent.fit(
    violent_cox,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

ctv_violent.print_summary()

In this case, there isn't a significant coefficient on African American's with High Scores.

In [ ]:
print(interaction_data.columns.tolist())

In [ ]:
from lifelines import CoxTimeVaryingFitter
import pandas as pd

interaction_data = data[["id", "start", "end", "event", "race", "score_text"]].copy()

interaction_data = pd.get_dummies(
    interaction_data,
    columns=["race", "score_text"],
    drop_first=True
)

# Interaction terms using the columns that actually exist
interaction_data["Asian_Low"] = (
    interaction_data["race_Asian"] *
    interaction_data["score_text_Low"]
)

interaction_data["Asian_Medium"] = (
    interaction_data["race_Asian"] *
    interaction_data["score_text_Medium"]
)

interaction_data["Caucasian_Low"] = (
    interaction_data["race_Caucasian"] *
    interaction_data["score_text_Low"]
)

interaction_data["Caucasian_Medium"] = (
    interaction_data["race_Caucasian"] *
    interaction_data["score_text_Medium"]
)

interaction_data["Hispanic_Low"] = (
    interaction_data["race_Hispanic"] *
    interaction_data["score_text_Low"]
)

interaction_data["Hispanic_Medium"] = (
    interaction_data["race_Hispanic"] *
    interaction_data["score_text_Medium"]
)

interaction_data["NativeAmerican_Low"] = (
    interaction_data["race_Native American"] *
    interaction_data["score_text_Low"]
)

interaction_data["NativeAmerican_Medium"] = (
    interaction_data["race_Native American"] *
    interaction_data["score_text_Medium"]
)

interaction_data["Other_Low"] = (
    interaction_data["race_Other"] *
    interaction_data["score_text_Low"]
)

interaction_data["Other_Medium"] = (
    interaction_data["race_Other"] *
    interaction_data["score_text_Medium"]
)

ctv_interaction = CoxTimeVaryingFitter()
ctv_interaction.fit(
    interaction_data,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

ctv_interaction.print_summary()

In [ ]:
from lifelines import CoxTimeVaryingFitter
import pandas as pd

# African-American only
violent_black = violent_data[violent_data["race"] == "African-American"].copy()
violent_black["score_factor"] = pd.Categorical(
    violent_black["score_text"],
    categories=["Low", "High", "Medium"]
)

violent_black_cox = violent_black[["id", "start", "end", "event", "score_factor"]].copy()
violent_black_cox = pd.get_dummies(
    violent_black_cox,
    columns=["score_factor"],
    drop_first=True
)

ctv_violent_black = CoxTimeVaryingFitter()
ctv_violent_black.fit(
    violent_black_cox,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

print("=== African-American defendants (violent model) ===")
ctv_violent_black.print_summary()


# Caucasian only
violent_white = violent_data[violent_data["race"] == "Caucasian"].copy()
violent_white["score_factor"] = pd.Categorical(
    violent_white["score_text"],
    categories=["Low", "High", "Medium"]
)

violent_white_cox = violent_white[["id", "start", "end", "event", "score_factor"]].copy()
violent_white_cox = pd.get_dummies(
    violent_white_cox,
    columns=["score_factor"],
    drop_first=True
)

ctv_violent_white = CoxTimeVaryingFitter()
ctv_violent_white.fit(
    violent_white_cox,
    id_col="id",
    start_col="start",
    stop_col="end",
    event_col="event"
)

print("\n=== Caucasian defendants (violent model) ===")
ctv_violent_white.print_summary()

The race-specific Cox models for the violent recidivism dataset show very similar patterns across groups.

For both African-American and Caucasian defendants, individuals classified as High risk have substantially higher hazard ratios compared to Low risk individuals. Medium risk individuals also show elevated hazard, though smaller than High risk.

This suggests that the COMPAS violent score consistently separates risk levels within each racial group.

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

# Match the R plot colors
colors = {
    "Low": "green",
    "Medium": "blue",
    "High": "red"
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# White defendants
white = violent_data[violent_data["race"] == "Caucasian"]

for group in ["Low", "Medium", "High"]:
    subset = white[white["score_text"] == group]
    kmf.fit(
        durations=subset["end"],
        event_observed=subset["event"],
        label=f"score_factor={group}"
    )
    kmf.plot_survival_function(ax=axes[0], color=colors[group])

axes[0].set_title("White defendants")
axes[0].set_xlabel("time")
axes[0].set_ylabel("surv")
axes[0].set_ylim(0, 1)
axes[0].legend(title="Risk level")

# Black defendants
black = violent_data[violent_data["race"] == "African-American"]

for group in ["Low", "Medium", "High"]:
    subset = black[black["score_text"] == group]
    kmf.fit(
        durations=subset["end"],
        event_observed=subset["event"],
        label=f"score_factor={group}"
    )
    kmf.plot_survival_function(ax=axes[1], color=colors[group])

axes[1].set_title("Black defendants")
axes[1].set_xlabel("time")
axes[1].set_ylabel("surv")
axes[1].set_ylim(0, 1)
axes[1].legend(title="Risk level")

plt.tight_layout()
plt.show()

The survival curves show clear separation between risk groups. Individuals classified as High risk have the lowest survival probabilities over time, while Low-risk individuals have the highest. This pattern is consistent across both African-American and Caucasian defendants, indicating that the COMPAS score meaningfully differentiates risk levels in both groups.


## Directions of the Racial Bias

The above analysis shows that the Compas algorithm does overpredict African-American defendant's future recidivism, but we haven't yet explored the direction of the bias. We can discover fine differences in overprediction and underprediction by comparing Compas scores across racial lines.

In [ ]:
# === Create binary prediction ===
violent_df = df.copy()

# Keep only Low / High
violent_df = violent_df[violent_df["v_score_text"].isin(["Low", "High"])].copy()

violent_df["pred_high"] = (violent_df["v_score_text"] == "High").astype(int)
violent_df["actual_recid"] = violent_df["two_year_recid"].astype(int)

# === Function ===
def show_confusion_table(data, title=""):
    table = pd.crosstab(data["actual_recid"], data["pred_high"])
    table = table.reindex(index=[0,1], columns=[0,1], fill_value=0)

    table.index = ["Survived", "Recidivated"]
    table.columns = ["Low", "High"]

    TN = table.loc["Survived", "Low"]
    FP = table.loc["Survived", "High"]
    FN = table.loc["Recidivated", "Low"]
    TP = table.loc["Recidivated", "High"]

    total = TN + FP + FN + TP

    fpr = FP / (FP + TN) * 100
    fnr = FN / (FN + TP) * 100
    specificity = TN / (TN + FP)
    sensitivity = TP / (TP + FN)

    print(title)
    print(table)
    print(f"Total: {total:.2f}")
    print(f"False positive rate: {fpr:.2f}")
    print(f"False negative rate: {fnr:.2f}")
    print(f"Specificity: {specificity:.2f}")
    print(f"Sensitivity: {sensitivity:.2f}")
    print()

    return {"FPR": fpr, "FNR": fnr}

# === All ===
all_results = show_confusion_table(violent_df, "All defendants")

# === Black ===
black_df = violent_df[violent_df["race"] == "African-American"]
black_results = show_confusion_table(black_df, "Black defendants")

# === White ===
white_df = violent_df[violent_df["race"] == "Caucasian"]
white_results = show_confusion_table(white_df, "White defendants")

# === Ratio ===
print("FPR ratio (Black / White):", black_results["FPR"] / white_results["FPR"])

Overall, the false positive rate is 32.35%.

In [ ]:
TN = 2681
FP = 1282

FPR = FP / (FP + TN)
print(FPR * 100)

That number is higher for African Americans at 44.85%.

Survived
Low = TN = 990
High = FP = 805

In [ ]:
FPR = (805 / (805 + 990)) * 100
print(FPR)

And lower for whites at 23.45%.

In [ ]:
FPR = (349 / (349 + 1139)) * 100
print(FPR)

Which means under COMPAS black defendants are 91% more likely to get a higher score and not go on to commit more crimes than white defendants after two year.

In [ ]:
44.85 / 23.45

COMPAS scores misclassify white reoffenders as low risk at 70.4% more often than black reoffenders.

In [ ]:
47.72 / 27.99

## Risk of Violent Recidivism

Compas also offers a score that aims to measure a persons risk of violent recidivism, which has a similar overall accuracy to the Recidivism score.

In [ ]:
# === Violent Recidivism ===
violent_df = df.copy()

# Low / Highだけ使う
violent_df = violent_df[violent_df["v_score_text"].isin(["Low", "High"])].copy()

# 予測
violent_df["pred_high"] = (violent_df["v_score_text"] == "High").astype(int)

# 実際
violent_df["actual_recid"] = violent_df["two_year_recid"].astype(int)

# === All defendants ===
all_results_v = show_confusion_table(violent_df, "All defendants (violent)")

# === Black defendants ===
black_v = violent_df[violent_df["race"] == "African-American"]
black_results_v = show_confusion_table(black_v, "Black defendants (violent)")

# === White defendants ===
white_v = violent_df[violent_df["race"] == "Caucasian"]
white_results_v = show_confusion_table(white_v, "White defendants (violent)")

# === Ratio ===
print("FPR ratio (Black / White):", black_results_v["FPR"] / white_results_v["FPR"])

In [ ]:
print(df.columns.tolist())
print(df[["v_score_text", "race", "two_year_recid"]].head())
print(violent_df["v_score_text"].value_counts())

Even moreso for Black defendants.

Black defendants are twice as likely to be false positives for a Higher violent score than white defendants.

In [ ]:
38.14 / 18.46

White defendants are 63% more likely to get a lower score and commit another crime than Black defendants.

In [ ]:
62.62 / 38.37

## Gender differences in Compas scores

In terms of underlying recidivism rates, we can look at gender specific Kaplan Meier estimates. There is a striking difference between women and men.

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

In [ ]:
!pip install lifelines

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

In [ ]:
data["score_factor"] = data["score_text"]

In [ ]:
female = data[data["sex"] == "Female"].copy()
male = data[data["sex"] == "Male"].copy()

In [ ]:
kmf = KaplanMeierFitter()

print("Female")
for score in ["Low", "Medium", "High"]:
    temp = female[female["score_factor"] == score]
    kmf.fit(
        durations=temp["end"] - temp["start"],
        event_observed=temp["event"],
        label=score
    )
    print(score, "survival at 730 days =", round(kmf.predict(730), 3))

print()

print("Male")
for score in ["Low", "Medium", "High"]:
    temp = male[male["score_factor"] == score]
    kmf.fit(
        durations=temp["end"] - temp["start"],
        event_observed=temp["event"],
        label=score
    )
    print(score, "survival at 730 days =", round(kmf.predict(730), 3))

As these plots show, the Compas score treats a High risk women the same as a Medium risk man.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Female
for score in ["High", "Low", "Medium"]:
    temp = female[female["score_factor"] == score]
    kmf.fit(
        durations=temp["end"] - temp["start"],
        event_observed=temp["event"],
        label=score
    )
    kmf.plot(ax=axes[0], ci_show=True)

axes[0].set_title("Female")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Survival Probability")

# Male
for score in ["High", "Low", "Medium"]:
    temp = male[male["score_factor"] == score]
    kmf.fit(
        durations=temp["end"] - temp["start"],
        event_observed=temp["event"],
        label=score
    )
    kmf.plot(ax=axes[1], ci_show=True)

axes[1].set_title("Male")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Survival Probability")

plt.tight_layout()
plt.show()

In [ ]:
print(len(df))
print(df["race"].value_counts(normalize=True)*100)
print((df["two_year_recid"]==1).mean()*100)

## Discussion of Code and Results

In this analysis, I translated the original COMPAS workflow into Python and replicated the main steps, including data preprocessing, exploratory analysis, and model estimation.

First, I applied several filtering conditions to improve data quality, reducing the dataset from 7,214 to 6,172 observations. This ensured that the analysis was conducted on a consistent and reliable subset of the data.

Exploratory analysis showed that African-American defendants make up the largest proportion of the sample (51.44%), followed by Caucasian defendants (34.07%), with an overall two-year recidivism rate of approximately 45.5%. The correlation between COMPAS scores and length of stay was weak (≈0.21), suggesting that higher scores are not strongly associated with time spent in custody.

Using logistic regression, I found that race remains a statistically significant predictor of COMPAS scores even after controlling for relevant variables. African-American defendants are approximately 1.45 times more likely than Caucasian defendants to receive a higher score. Age also has a strong effect, with individuals under 25 being about 2.5 times more likely to receive a higher score compared to those aged 25–45.

A similar pattern appears in the violent recidivism model. African-American defendants are about 1.77 times more likely to receive a higher violent risk score, and individuals under 25 are more than seven times as likely to receive a higher score. These results indicate that demographic factors continue to play a significant role across different model specifications.

In terms of predictive performance, the Cox model shows that higher COMPAS scores are associated with a higher hazard of recidivism, indicating that the model captures meaningful risk patterns. However, classification analysis reveals that the false positive rate (FPR) is substantially higher for African-American defendants than for Caucasian defendants, suggesting unequal error distribution across groups.

From a broader perspective, these findings highlight a key limitation of evaluating models based solely on overall accuracy. The observed disparities in error rates suggest that the model may violate fairness criteria such as equalized odds, where error rates are expected to be similar across groups. This raises concerns about the use of such models in high-stakes decision-making contexts, including bail and sentencing.

Overall, while the COMPAS model demonstrates predictive validity, it also exhibits systematic differences across demographic groups. This suggests that model evaluation should incorporate both predictive performance and fairness considerations to ensure more equitable outcomes in practice.